# Persistent Colab workspace

Keep files and a saved Python environment in Google Drive, then restore them on a new CPU or GPU runtime.

**Start:** choose CPU for ordinary development, or select a GPU when needed. Run cells 1 and 2 before importing your own packages. Grant Drive access if Google asks. Use the same project name on future visits.

**Finish:** save model/training checkpoints, run **Save environment**, then **Flush Drive**. Finally use **Runtime → Disconnect and delete runtime** to release compute.

Drive preserves files and saved environment archives. A released VM loses RAM, GPU memory, running programs and unsaved changes. A Python/base-image change requires an explicit environment rebuild. Google Drive capacity does not enlarge the runtime disk.


In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. Restore workspace and Python environment
PROJECT = 'default'  # @param {type:"string"}
exec('"""Restore a Python workspace from Drive without pretending the VM is persistent."""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport site\nimport socket\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom datetime import datetime, timezone\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with temporary.open("w") as stream:\n        json.dump(value, stream, indent=2)\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef runtime_signature():\n    # Base packages are inherited, rather than copied into each environment.\n    # A changed base image requires an explicit rebuild, never a silent restore.\n    packages = json.loads(subprocess.check_output(\n        [sys.executable, "-m", "pip", "list", "--format=json"], text=True))\n    normalized = sorted((p["name"].lower().replace("_", "-"), p["version"])\n                        for p in packages)\n    return {\n        "python": list(sys.version_info[:2]),\n        "machine": platform.machine(),\n        "os": platform.system(),\n        "libc": list(platform.libc_ver()),\n        "base_packages_sha256": hashlib.sha256(\n            json.dumps(normalized).encode()).hexdigest(),\n        "base_packages": [list(item) for item in normalized],\n    }\n\n\nPLATFORM_KEYS = ("python", "machine", "os", "libc")\n\n\ndef package_drift(saved, current):\n    """Describe base package differences as sorted \'name: old -> new\' lines."""\n    before, after = dict(map(tuple, saved)), dict(map(tuple, current))\n    return [f"{name}: {before.get(name, \'absent\')} -> {after.get(name, \'absent\')}"\n            for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]\n\n\nLOCK_HEARTBEAT = 60\nLOCK_STALE = 600\n\n\ndef _utc():\n    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n\n\ndef _lock_age(record):\n    heartbeat = datetime.strptime(record["heartbeat_utc"], "%Y-%m-%dT%H:%M:%SZ")\n    return (datetime.now(timezone.utc) - heartbeat.replace(tzinfo=timezone.utc)).total_seconds()\n\n\ndef _heartbeat_name(path):\n    return "colab-workspace-lock:" + str(path)\n\n\ndef _start_heartbeat(path):\n    name = _heartbeat_name(path)\n    if any(thread.name == name and thread.is_alive() for thread in threading.enumerate()):\n        return\n    stop = threading.Event()\n\n    def beat():\n        while not stop.wait(LOCK_HEARTBEAT):\n            try:\n                record = json.loads(path.read_text())\n                if record.get("host") != socket.gethostname():\n                    return\n                record.update(heartbeat_utc=_utc(), pid=os.getpid())\n                write_json(path, record)\n            except (OSError, ValueError):\n                # Storage was unmounted or the lock was removed; never recreate it.\n                return\n\n    thread = threading.Thread(target=beat, name=name, daemon=True)\n    thread.stop = stop\n    thread.start()\n\n\ndef acquire_lock(root, session=None, force=False):\n    """Claim a workspace for this runtime; detects, but cannot prevent, racing writers."""\n    path = Path(root) / "environment" / "LOCK"\n    host = socket.gethostname()\n    record = {"session": session, "host": host, "pid": os.getpid(),\n              "created_utc": _utc(), "heartbeat_utc": _utc()}\n    try:\n        with path.open("xb") as stream:\n            stream.write(json.dumps(record, indent=2).encode() + b"\\n")\n            stream.flush()\n            os.fsync(stream.fileno())\n    except FileExistsError:\n        try:\n            current = json.loads(path.read_text())\n        except (OSError, ValueError):\n            current = None\n        if not force and (current is None or current.get("host") != host):\n            if current is None:\n                raise RuntimeError(f"Unreadable workspace lock {path}. Retry with --force-unlock "\n                                   "(notebook: ws.prepare(force_unlock=True)) if no runtime uses it.") from None\n            age = _lock_age(current)\n            owner = f"session {current.get(\'session\')!r} on host {current.get(\'host\')!r}"\n            if age > LOCK_STALE:\n                raise RuntimeError(f"Workspace lock held by {owner} is stale (no heartbeat for {age / 60:.0f} min). "\n                                   "If that runtime is gone, retry with --force-unlock "\n                                   "(notebook: ws.prepare(force_unlock=True)).") from None\n            raise RuntimeError(f"Workspace is in use by {owner} (heartbeat {age:.0f}s ago). "\n                               "Use one writer per project; finish or flush the other runtime first.") from None\n        if current and current.get("host") == host:\n            record["created_utc"] = current.get("created_utc", record["created_utc"])\n        write_json(path, record)\n    _start_heartbeat(path)\n    return path\n\n\ndef lock_held_elsewhere(root):\n    path = Path(root) / "environment" / "LOCK"\n    try:\n        return json.loads(path.read_text()).get("host") != socket.gethostname()\n    except FileNotFoundError:\n        return False\n    except ValueError:\n        return True\n\n\ndef release_lock(root):\n    """Stop the heartbeat and remove this runtime\'s lock before storage is flushed."""\n    path = Path(root) / "environment" / "LOCK"\n    for thread in threading.enumerate():\n        if thread.name == _heartbeat_name(path):\n            thread.stop.set()\n            thread.join()\n    if path.exists() and not lock_held_elsewhere(root):\n        state = rclone_state()\n        if state and path.resolve().is_relative_to(Path(state["mount_point"]).resolve()):\n            # Deleting a file whose rewrite is still queued drops only the queued\n            # upload; the previous object would stay on the remote.\n            wait_rclone_uploads(state)\n        path.unlink()\n\n\ndef prune_snapshots(root, keep, apply=False):\n    """Remove old snapshot pairs, always keeping the newest ones and the latest pointer."""\n    if keep < 1:\n        raise ValueError("keep must be at least 1.")\n    snapshots = Path(root) / "environment" / "snapshots"\n    latest = Path(root) / "environment" / "latest.json"\n    current = json.loads(latest.read_text())["created_utc"] if latest.exists() else None\n    stamps = sorted({path.name.split(".", 1)[0] for path in snapshots.iterdir()\n                     if path.name.endswith((".tar.gz", ".json"))}, reverse=True)\n    retained = set(stamps[:keep]) | {current}\n    removed = [stamp for stamp in stamps if stamp not in retained]\n    for stamp in removed:\n        for path in (snapshots / (stamp + ".tar.gz"), snapshots / (stamp + ".json")):\n            if path.exists():\n                print(("Deleted " if apply else "Would delete ") + str(path))\n                if apply:\n                    path.unlink()\n    print(f"Kept {len(stamps) - len(removed)} snapshot(s); {\'deleted\' if apply else \'would delete\'} {len(removed)}.")\n    if removed and not apply:\n        print("Dry run only. Repeat with --yes to delete.")\n    return removed\n\n\nRCLONE_CONF = Path("/root/.config/rclone/rclone.conf")\nRCLONE_STATE = Path("/root/.config/colab-workspace/rclone-mount.json")\nRCLONE_RC = "127.0.0.1:5572"\nRCLONE_LOG = Path("/tmp/colab-workspace-rclone.log")\n# Colab ships FUSE 2 only; rclone execs fusermount3, which accepts the same protocol.\nFUSE_SHIM = Path("/opt/colab-workspace/bin")\n\n\ndef _rclone(*arguments, config=RCLONE_CONF):\n    return subprocess.run(["rclone", "--config", str(config), *map(str, arguments)],\n                          check=True, capture_output=True, text=True, stdin=subprocess.DEVNULL)\n\n\ndef _remote_join(remote, relative):\n    relative = str(relative).strip("/")\n    if not relative or relative == ".":\n        return remote\n    return remote + relative if remote.endswith(":") else remote + "/" + relative\n\n\ndef rclone_state():\n    return json.loads(RCLONE_STATE.read_text()) if RCLONE_STATE.exists() else None\n\n\ndef _install_rclone():\n    if shutil.which("rclone"):\n        return\n    script = subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True,\n                            capture_output=True, text=True, stdin=subprocess.DEVNULL)\n    if script.returncode and not shutil.which("rclone"):\n        apt = subprocess.run("apt-get update -qq && apt-get install -y -qq rclone", shell=True,\n                             capture_output=True, text=True, stdin=subprocess.DEVNULL)\n        if apt.returncode:\n            raise RuntimeError("Could not install rclone:\\n" + script.stderr[-1000:] + apt.stderr[-1000:])\n\n\ndef _fuse_environment():\n    environment = os.environ.copy()\n    if not shutil.which("fusermount3"):\n        fusermount = shutil.which("fusermount")\n        if fusermount is None or not Path("/dev/fuse").exists():\n            raise RuntimeError("FUSE is unavailable on this runtime (/dev/fuse or fusermount missing).")\n        FUSE_SHIM.mkdir(parents=True, exist_ok=True)\n        shim = FUSE_SHIM / "fusermount3"\n        if not shim.is_symlink():\n            shim.symlink_to(fusermount)\n        environment["PATH"] = str(FUSE_SHIM) + os.pathsep + environment["PATH"]\n    return environment\n\n\ndef mount_rclone(remote, mount_point, staged_config, timeout=120):\n    """Install a staged rclone config and mount the remote; remove the config on failure."""\n    mount_point = Path(mount_point).resolve()\n    state = rclone_state()\n    if os.path.ismount(mount_point):\n        Path(staged_config).unlink(missing_ok=True)\n        if state and state["remote"] == remote and state["mount_point"] == str(mount_point):\n            print("rclone remote is already mounted:", mount_point)\n            return state\n        raise RuntimeError(f"{mount_point} is already mounted by something else.")\n    try:\n        RCLONE_CONF.parent.mkdir(parents=True, exist_ok=True)\n        os.chmod(RCLONE_CONF.parent, 0o700)\n        shutil.move(staged_config, RCLONE_CONF)\n        os.chmod(RCLONE_CONF, 0o600)\n        _install_rclone()\n        environment = _fuse_environment()\n        mount_point.mkdir(parents=True, exist_ok=True)\n        if any(mount_point.iterdir()):\n            raise RuntimeError(f"{mount_point} is not empty; choose an empty mount point.")\n        # A new session keeps the daemon alive when the kernel restarts.\n        daemon = subprocess.run(\n            ["rclone", "--config", str(RCLONE_CONF), "mount", remote, str(mount_point),\n             "--vfs-cache-mode", "writes", "--rc", "--rc-addr", RCLONE_RC, "--rc-no-auth",\n             "--daemon", "--log-file", str(RCLONE_LOG), "--log-level", "NOTICE"],\n            env=environment, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,\n            stderr=subprocess.DEVNULL, start_new_session=True)\n        deadline = time.monotonic() + timeout\n        while not os.path.ismount(mount_point):\n            if daemon.returncode or time.monotonic() > deadline:\n                log = RCLONE_LOG.read_text()[-2000:] if RCLONE_LOG.exists() else ""\n                raise RuntimeError("rclone mount did not become ready:\\n" + log)\n            time.sleep(1)\n        os.listdir(mount_point)\n        state = {"remote": remote, "mount_point": str(mount_point),\n                 "config": str(RCLONE_CONF), "rc": RCLONE_RC}\n        RCLONE_STATE.parent.mkdir(parents=True, exist_ok=True)\n        write_json(RCLONE_STATE, state)\n        print("Mounted rclone remote at", mount_point)\n        return state\n    except BaseException:\n        Path(staged_config).unlink(missing_ok=True)\n        RCLONE_CONF.unlink(missing_ok=True)\n        raise\n\n\ndef _md5(path):\n    digest = hashlib.md5()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef wait_rclone_uploads(state, timeout=1800, poll=2):\n    """Block until the rclone VFS has no queued or running uploads."""\n    url = "http://" + state["rc"] + "/"\n    deadline = time.monotonic() + timeout\n    while True:\n        cache = json.loads(_rclone("rc", "vfs/stats", "--url", url, config=state["config"]).stdout)["diskCache"]\n        if cache.get("erroredFiles"):\n            raise RuntimeError(f"rclone reports {cache[\'erroredFiles\']} file(s) that failed to upload; see {RCLONE_LOG}.")\n        if cache["uploadsInProgress"] == 0 and cache["uploadsQueued"] == 0:\n            return\n        if time.monotonic() > deadline:\n            raise RuntimeError("Timed out waiting for rclone uploads; the remote is still mounted.")\n        time.sleep(poll)\n\n\ndef flush_rclone(workspace_roots=(), timeout=1800, poll=2):\n    """Wait for VFS uploads, verify the remote copy, unmount and delete the config."""\n    state = rclone_state()\n    if state is None:\n        raise RuntimeError("No rclone mount was created by colab-workspace on this runtime.")\n    mount_point = Path(state["mount_point"]).resolve()\n    config = Path(state["config"])\n    deadline = time.monotonic() + timeout\n    wait_rclone_uploads(state, timeout, poll)\n    for root in map(Path, workspace_roots):\n        remote_root = _remote_join(state["remote"], root.relative_to(mount_point))\n        remote_files = _rclone("lsf", "--files-only", _remote_join(remote_root, "environment"),\n                               config=config).stdout.split("\\n")\n        if "LOCK" in remote_files:\n            remote_lock = json.loads(_rclone("cat", _remote_join(remote_root, "environment/LOCK"),\n                                             config=config).stdout)\n            if remote_lock.get("host") == socket.gethostname():\n                raise RuntimeError("This runtime\'s lock is still on the remote; the remote is still mounted.")\n        latest = root / "environment" / "latest.json"\n        if not latest.exists():\n            continue\n        local_pointer = json.loads(latest.read_text())\n        remote_pointer = json.loads(_rclone(\n            "cat", _remote_join(remote_root, "environment/latest.json"), config=config).stdout)\n        if remote_pointer != local_pointer:\n            raise RuntimeError("Remote latest.json differs from the local copy; the remote is still mounted.")\n        archive = root / "environment" / "snapshots" / local_pointer["archive"]\n        listing = json.loads(_rclone(\n            "lsjson", "--stat", "--hash", "--hash-type", "md5",\n            _remote_join(remote_root, "environment/snapshots/" + local_pointer["archive"]),\n            config=config).stdout)\n        remote_md5 = (listing.get("Hashes") or {}).get("md5")\n        if listing.get("Size") != archive.stat().st_size or (remote_md5 and remote_md5 != _md5(archive)):\n            raise RuntimeError("Remote snapshot archive does not match; the remote is still mounted.")\n        print("Verified remote snapshot:", local_pointer["archive"])\n    if Path.cwd().is_relative_to(mount_point):\n        os.chdir("/")\n    fusermount = shutil.which("fusermount3") or shutil.which("fusermount")\n    if fusermount is None:\n        raise RuntimeError("fusermount is unavailable; the remote is still mounted.")\n    subprocess.run([fusermount, "-u", str(mount_point)], check=True,\n                   capture_output=True, text=True, stdin=subprocess.DEVNULL)\n    while os.path.ismount(mount_point):\n        if time.monotonic() > deadline:\n            raise RuntimeError("Timed out unmounting the rclone remote.")\n        time.sleep(poll)\n    # rclone may rewrite the config when refreshing a token; remove every copy.\n    for path in config.parent.glob(config.name + "*"):\n        path.unlink()\n    RCLONE_STATE.unlink()\n    print("rclone uploads verified; remote unmounted and credentials removed.")\n\n\nclass Workspace:\n    """Use durable files on Drive and executable packages on the current VM."""\n\n    def __init__(self, project="default", drive_root=None, local_root=None, session=None):\n        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", project):\n            raise ValueError("Project must use letters, digits, underscores or hyphens.")\n        default_drive = Path("/content/drive/MyDrive")\n        if drive_root is None and not default_drive.is_dir():\n            raise RuntimeError("Mount Google Drive before creating a workspace.")\n        self.project = project\n        self.session = session\n        # Subprocesses run with cwd=self.root, so relative paths must be absolute.\n        self.root = (Path(drive_root or default_drive / "PersistentColab" / "workspaces") / project).resolve()\n        self.local = (Path(local_root or "/content/persistent-colab") / project).resolve()\n        self.venv = self.local / ".venv"\n        self.python = self.venv / "bin" / "python"\n        self.env_dir = self.root / "environment"\n        self.snapshots = self.env_dir / "snapshots"\n        self.latest = self.env_dir / "latest.json"\n        self.signature = runtime_signature()\n        self.activated = False\n        for folder in ("src", "data", "models", "outputs", "checkpoints", "environment/snapshots"):\n            (self.root / folder).mkdir(parents=True, exist_ok=True)\n        self.local.mkdir(parents=True, exist_ok=True)\n        # Model files are durable, tokens remain in the ephemeral runtime.\n        os.environ["HF_HUB_CACHE"] = str(self.root / "models" / "huggingface")\n        os.environ["HF_TOKEN_PATH"] = str(self.local / "hf-token")\n        os.environ["TORCH_HOME"] = str(self.root / "models" / "torch")\n\n    def prepare(self, restore=True, force_unlock=False, allow_base_drift=False):\n        """Claim the workspace, recreate executable paths and restore the latest compatible snapshot."""\n        acquire_lock(self.root, self.session, force_unlock)\n        try:\n            return self._prepare(restore, allow_base_drift)\n        except BaseException:\n            release_lock(self.root)\n            raise\n\n    def _compatibility(self, saved, allow_base_drift):\n        """Return base package drift accepted for this restore, or raise."""\n        rebuild = ("Review environment/latest.json, then explicitly run ws.rebuild() to "\n                   "reinstall the saved requirements on this image. Old snapshots are kept.")\n        if any(saved.get(key) != self.signature.get(key) for key in PLATFORM_KEYS):\n            raise RuntimeError("The Colab Python/base image changed (Python, architecture or libc). "\n                               "The saved environment was not loaded. " + rebuild)\n        if saved.get("base_packages_sha256") == self.signature.get("base_packages_sha256"):\n            return []\n        if "base_packages" not in saved:\n            raise RuntimeError("The Colab Python/base image changed. The saved environment was not loaded. " + rebuild)\n        drift = package_drift(saved["base_packages"], self.signature["base_packages"])\n        if not allow_base_drift:\n            raise RuntimeError("The Colab base image changed; the saved environment was not loaded. "\n                               "Changed base packages:\\n  " + "\\n  ".join(drift) +\n                               "\\nRetry with allow_base_drift=True (CLI: --allow-base-drift) to restore and "\n                               "validate it with pip check, or " + rebuild)\n        print("Base packages differ from the snapshot:\\n  " + "\\n  ".join(drift))\n        return drift\n\n    def _dependency_problems(self, python):\n        result = subprocess.run([str(python), "-m", "pip", "check"], capture_output=True, text=True)\n        return set(result.stdout.splitlines()) if result.returncode else set()\n\n    def _prepare(self, restore, allow_base_drift):\n        ready = self.local / "environment-ready.json"\n        if self.python.exists():\n            if ready.exists() and json.loads(ready.read_text()) == self.signature:\n                return self\n            raise RuntimeError("Incomplete local environment. Start a fresh runtime to restore safely.")\n        snapshot = json.loads(self.latest.read_text()) if restore and self.latest.exists() else None\n        drift = self._compatibility(snapshot["runtime"], allow_base_drift) if snapshot else []\n        if snapshot and snapshot["local_path"] != str(self.venv):\n            raise RuntimeError("Snapshot executable paths differ. Use the original project name and runtime path.")\n        # Colab\'s image has pip but its ensurepip bootstrap is unavailable.\n        # Inherit pip from the base image; its install target is still this venv.\n        subprocess.run([sys.executable, "-m", "venv", "--without-pip",\n                        "--system-site-packages", str(self.venv)], check=True)\n        if snapshot:\n            archive_name = snapshot["archive"]\n            if Path(archive_name).name != archive_name:\n                raise ValueError("Invalid snapshot archive name.")\n            source = self.snapshots / archive_name\n            local_archive = self.local / "restore.tar.gz"\n            shutil.copyfile(source, local_archive)\n            if sha256(local_archive) != snapshot["sha256"]:\n                shutil.rmtree(self.venv)\n                local_archive.unlink()\n                raise RuntimeError("Snapshot checksum mismatch; environment was not restored.")\n            try:\n                with tarfile.open(local_archive, "r:gz") as archive:\n                    archive.extractall(self.venv, filter="data")\n            except Exception:\n                # This directory is an agent-owned, disposable VM environment only.\n                shutil.rmtree(self.venv)\n                raise\n            finally:\n                local_archive.unlink(missing_ok=True)\n            if drift:\n                # Colab images already ship some unmet requirements; only new ones fail.\n                problems = self._dependency_problems(self.python) - self._dependency_problems(sys.executable)\n                if problems:\n                    shutil.rmtree(self.venv)\n                    raise RuntimeError("Restored environment conflicts with the new base image; it was removed:\\n  "\n                                       + "\\n  ".join(sorted(problems)))\n                print("pip check found no new conflicts with the changed base image.")\n            print("Restored Python environment:", archive_name)\n        else:\n            print("Created a new Python environment; Colab base packages are available.")\n        write_json(ready, self.signature)\n        print("Persistent workspace:", self.root)\n        return self\n\n    def run(self, code, capture=False):\n        """Run Python in the saved environment; persistent workspace is its cwd."""\n        return subprocess.run([str(self.python), "-c", code], cwd=self.root,\n                              check=True, text=True, capture_output=capture)\n\n    def install(self, *packages):\n        """Install into the environment and immediately persist its new snapshot."""\n        if not packages or any(p.startswith("-") for p in packages):\n            raise ValueError("Pass package specifications, such as \'humanize==4.15.0\'.")\n        subprocess.run([str(self.python), "-m", "pip", "install", *packages], check=True)\n        result = self.save_environment()\n        importlib.invalidate_caches()\n        print("New packages are available in fresh imports; restart before replacing loaded packages.")\n        return result\n\n    def save_environment(self):\n        """Commit an immutable archive, publishing the pointer only after readback."""\n        if not self.python.exists():\n            raise RuntimeError("Run ws.prepare() first.")\n        if lock_held_elsewhere(self.root):\n            raise RuntimeError("Another runtime holds this workspace\'s lock; the environment was not saved.")\n        requirements = subprocess.check_output(\n            [str(self.python), "-m", "pip", "freeze", "--local"], text=True)\n        # Do not persist credentials embedded in installation URLs.\n        if "://" in requirements or "-e " in requirements or " @ " in requirements:\n            raise RuntimeError("Use versioned index packages; save local/editable source under ws.root / \'src\'.")\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]\n        name = stamp + ".tar.gz"\n        with tempfile.TemporaryDirectory(dir=self.local) as temporary:\n            archive_path = Path(temporary) / name\n            with tarfile.open(archive_path, "w:gz", compresslevel=1) as archive:\n                for directory in ("lib", "bin", "include"):\n                    source = self.venv / directory\n                    if source.exists():\n                        def exclude_interpreter(info):\n                            if info.name.startswith("bin/python"):\n                                return None\n                            if info.issym() and (self.venv / info.name).resolve() == Path(sys.executable).resolve():\n                                return None\n                            if info.issym() and Path(info.linkname).is_absolute():\n                                raise RuntimeError("Environment contains an external symlink: " + info.name)\n                            if "__pycache__" in Path(info.name).parts:\n                                return None\n                            return info\n                        archive.add(source, arcname=directory, filter=exclude_interpreter)\n            expected = sha256(archive_path)\n            destination = self.snapshots / name\n            with archive_path.open("rb") as source, destination.open("xb") as target:\n                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)\n                target.flush()\n                os.fsync(target.fileno())\n            if sha256(destination) != expected:\n                raise RuntimeError("Drive snapshot readback failed; previous snapshot remains current.")\n        snapshot = {"schema": 2, "archive": name, "sha256": expected,\n                    "created_utc": stamp, "runtime": self.signature,\n                    "requirements": requirements, "local_path": str(self.venv)}\n        write_json(self.snapshots / (stamp + ".json"), snapshot)\n        write_json(self.env_dir / "latest.json", snapshot)\n        print("Environment saved and checksum verified:", name)\n        return snapshot\n\n    def rebuild(self, force_unlock=False):\n        """Explicitly reinstall recorded packages when Colab\'s base image changed."""\n        if self.activated:\n            raise RuntimeError("Restart the kernel before rebuilding an activated environment.")\n        previous = json.loads(self.latest.read_text())\n        if self.venv.exists():\n            raise RuntimeError("Use a fresh runtime for rebuilding; current environment is preserved.")\n        self.prepare(restore=False, force_unlock=force_unlock)\n        lock = self.local / "requirements.txt"\n        lock.write_text(previous["requirements"])\n        if previous["requirements"].strip():\n            subprocess.run([str(self.python), "-m", "pip", "install", "-r", str(lock)], check=True)\n        return self.save_environment()\n\n    def activate(self):\n        """Expose this environment to fresh imports in ordinary notebook cells."""\n        paths = json.loads(subprocess.check_output(\n            [str(self.python), "-c", "import site,json; print(json.dumps(site.getsitepackages()))"], text=True))\n        own = [p for p in paths if Path(p).is_relative_to(self.venv)]\n        for path in reversed(own):\n            site.addsitedir(path)\n            sys.path.remove(path)\n            sys.path.insert(0, path)\n        os.environ["PATH"] = str(self.venv / "bin") + os.pathsep + os.environ["PATH"]\n        self.activated = True\n        importlib.invalidate_caches()\n        os.chdir(self.root)\n        print("Notebook imports use the saved environment. Install through ws.install(...).")\n        return self\n\n    def write_checkpoint(self, state, name="session.json"):\n        """Save explicit JSON state; arbitrary RAM/GPU state is never implicit."""\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        write_json(self.root / "checkpoints" / name, state)\n        return self.root / "checkpoints" / name\n\n    def read_checkpoint(self, name="session.json"):\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        path = self.root / "checkpoints" / name\n        return json.loads(path.read_text()) if path.exists() else None\n\n    def flush(self):\n        """Flush storage writes before the user releases the VM."""\n        release_lock(self.root)\n        state = rclone_state()\n        if state and self.root.is_relative_to(Path(state["mount_point"]).resolve()):\n            flush_rclone([self.root])\n        else:\n            from google.colab import drive\n            drive.flush_and_unmount()\n        print("Storage writes flushed. You may now disconnect and delete this runtime.")\n')
ws = Workspace(PROJECT)
ws.prepare()
ws.activate()
print('Previous saved state:', ws.read_checkpoint())


## Work here

Your current directory is the durable Drive workspace. Use `ws.root / 'src'`, `'data'`, `'models'`, `'outputs'`, and `'checkpoints'` for files that must survive.

Install additional packages with `ws.install('package==version')`; it saves the environment immediately. The environment inherits Colab's preinstalled AI libraries. Do not use `%pip` or `!pip` for packages you want this environment to retain. Use `ws.run(code)` or `subprocess.run([str(ws.python), ...])` for an independent Python process and for packages already imported by the notebook kernel. Installing a replacement for an already loaded package requires restarting the kernel and running cells 1 and 2 again.

Hugging Face model downloads and Torch downloads use the Drive model directory. Authentication tokens stay on the temporary runtime. Keep secrets in Colab Secrets; never in the notebook, workspace or checkpoints.

For many small files, unpack a dataset on the runtime's local disk for speed, retaining its original archive on Drive. Only files actually saved back to Drive survive. Concurrent writers to the same project are unsupported.


In [ ]:
# @title Optional: install and save packages
# Uncomment and change the package list when needed.
# ws.install('humanize==4.15.0')


In [ ]:
# @title Your Python / AI code
from pathlib import Path
print('Working directory:', Path.cwd())
print('Persistent output directory:', ws.root / 'outputs')
print('Environment Python:', ws.python)
# Write your code in this cell or add more cells here.


## Save progress explicitly

Use `ws.write_checkpoint({'step': 100, ...})` for JSON state; load it with `ws.read_checkpoint()`. This is application-defined state, not an automatic copy of Python variables.

For Hugging Face Trainer, set `output_dir=str(ws.root / 'checkpoints' / 'training')`, save checkpoints regularly, and use `trainer.train(resume_from_checkpoint=...)` to resume a complete checkpoint. Saving model weights alone does not preserve optimizer, scheduler and training progress. Keep the model, dataset and training configuration compatible when resuming.

System installations (`apt`), OS services and driver changes are not included in the Python snapshot. Put necessary setup commands in explicit notebook cells and rerun them on a new VM.


In [ ]:
# @title Save environment
saved_environment = ws.save_environment()
print('Saved at:', saved_environment['created_utc'])


## Before releasing compute

Check **FLUSH_DRIVE** and run the next cell **only when finished**. It releases the workspace lock, then flushes and unmounts Drive. Then disconnect and delete the runtime using the Runtime menu. To continue working after unmounting, mount Drive again.

If Colab reports that its Python/base image changed, restart with a fresh runtime, run cell 1 and instantiate `ws` using cell 2; after the compatibility error, run `ws.rebuild()` and `ws.activate()` in a separate cell. This explicitly reinstalls the recorded package versions and saves a new snapshot; previous snapshots remain on Drive. Reinstallation can fail when a package no longer supports the new image. When only base packages changed, `ws.prepare(allow_base_drift=True)` restores the saved environment instead and removes it again if `pip check` reports a new conflict. CPU and GPU runtimes use different base images.

Cell 2 claims the workspace for this runtime. If another runtime ended without flushing, the error reports a stale lock after 10 minutes; once you are sure that runtime is gone, use `ws.prepare(force_unlock=True)`.


In [ ]:
# @title Flush Drive (run manually when finished)
FLUSH_DRIVE = False  # @param {type:"boolean"}
if FLUSH_DRIVE:
    ws.flush()
else:
    print('Check FLUSH_DRIVE and run this cell when you are finished.')
